# UTS: Sistem Klasifikasi Tutupan Lahan Jawa Timur

## 1. Business Understanding (Dapat Informasi)

Pemetaan permukaan bumi menggunakan citra satelit untuk tata ruang wilayah. Menggunakan Band 8A, 11, dan 4 untuk secara akurat membedakan lahan basah (Hutan Mangrove) dengan Hutan Daratan biasa.

Sistem ini memetakan lima kelas tutupan lahan: **Bangunan, Lahan Pertanian, Perairan, Hutan, dan Hutan Mangrove**. Band 8A (NIR sempit), Band 11 (SWIR), dan Band 4 (merah) digunakan sebagai fitur spektral Sentinel-2. Band 11 sensitif terhadap kandungan air pada vegetasi dan tanah, sehingga membantu membedakan mangrove di kawasan pasang-surut yang basah dari hutan daratan. Nilai spektral tetap perlu dipadukan dengan sampel latihan yang representatif; satu band saja tidak menjamin pemisahan kelas.

In [ ]:
%pip -q install earthengine-api geopandas scikit-learn folium requests

import json
import os
import re
import requests
import ee
import pandas as pd
import geopandas as gpd
import folium
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Jalankan autentikasi satu kali jika diminta; Colab akan membuka alur login Google.
try:
    ee.Initialize()
except ee.EEException:
    ee.Authenticate()
    ee.Initialize()

print("Google Earth Engine siap digunakan.")

Note: you may need to restart the kernel to use updated packages.


## 2. Tahap "Ambil" & "Proses" (GEE & Data Sentinel-2)

Mengambil komposit Sentinel-2 L2A (surface reflectance) untuk Jawa Timur, menyaring awan berdasarkan Scene Classification Layer (SCL), lalu memilih hanya B8A, B11, dan B4. Rentang tanggal memakai batas akhir eksklusif.

In [1]:
# Parameter dapat diubah sesuai musim/tujuan pemetaan.
START_DATE = '2023-01-01'
END_DATE = '2024-01-01'  # tanggal akhir tidak termasuk
MAX_CLOUD_PERCENT = 30
bands = ['B8A', 'B11', 'B4']

# Batas administrasi level 1 GAUL: Jawa Timur tercatat sebagai East Java.
roi = (ee.FeatureCollection('FAO/GAUL/2015/level1')
       .filter(ee.Filter.eq('ADM0_NAME', 'Indonesia'))
       .filter(ee.Filter.eq('ADM1_NAME', 'East Java')))
if roi.size().getInfo() == 0:
    raise ValueError("AOI Jawa Timur tidak ditemukan pada FAO/GAUL/2015/level1.")

def mask_s2_clouds(image):
    # SCL konsisten digunakan untuk masking awan/bayangan pada koleksi L2A.
    scl = image.select('SCL')
    # Pertahankan hanya vegetasi, tanah terbuka, dan air; kelas lain mencakup awan/bayangan/no-data.
    clear = scl.eq(4).Or(scl.eq(5)).Or(scl.eq(6))
    return image.updateMask(clear).select(bands).divide(10000)

s2_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                 .filterBounds(roi)
                 .filterDate(START_DATE, END_DATE)
                 .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', MAX_CLOUD_PERCENT)))
image_count = s2_collection.size().getInfo()
if image_count == 0:
    raise ValueError('Tidak ada citra Sentinel-2 pada AOI dan rentang tanggal tersebut.')

selected_image = (s2_collection.map(mask_s2_clouds)
                  .median()
                  .select(bands)
                  .clip(roi.geometry()))
print(f'{image_count} scene Sentinel-2 ditemukan; band yang digunakan: {bands}.')

NameError: name 'ee' is not defined

### Ekstraksi Nilai Pixel dari GeoJSON

Simpan GeoJSON digitasi sebagai `data_training.geojson` di direktori kerja (atau unggah ke Colab). Properti kelas harus bernama `label` dan berisi nama kelas yang tercantum di bawah, atau ID 0–4. Poligon diproyeksikan ke WGS84, diubah menjadi fitur Earth Engine, lalu sampel piksel diekspor sebagai CSV agar hasil besar tidak ditarik melalui `getInfo()`.

In [ ]:
geojson_path = 'data_training.geojson'
LABEL_COLUMN = 'label'
class_names = {
    0: 'Bangunan',
    1: 'Lahan Pertanian',
    2: 'Perairan',
    3: 'Hutan',
    4: 'Hutan Mangrove',
}
name_to_id = {name.casefold(): class_id for class_id, name in class_names.items()}

if not os.path.exists(geojson_path):
    raise FileNotFoundError(
        f'{geojson_path} tidak ditemukan. Unggah/salin data_training.geojson ke direktori kerja notebook.'
    )

training_gdf = gpd.read_file(geojson_path)
if training_gdf.empty or training_gdf.geometry.isna().any():
    raise ValueError('GeoJSON training kosong atau memiliki geometri yang hilang.')
if LABEL_COLUMN not in training_gdf.columns:
    raise ValueError(f"Properti kelas '{LABEL_COLUMN}' tidak ditemukan. Ubah LABEL_COLUMN sesuai GeoJSON.")
if training_gdf.crs is None:
    training_gdf = training_gdf.set_crs('EPSG:4326')
else:
    training_gdf = training_gdf.to_crs('EPSG:4326')

def class_id_from_value(value):
    if pd.isna(value):
        raise ValueError('Ada label kosong pada GeoJSON training.')
    if isinstance(value, (int, float)) and float(value).is_integer():
        class_id = int(value)
        if class_id in class_names:
            return class_id
    normalized = re.sub(r'\s+', ' ', str(value).strip()).casefold()
    if normalized in name_to_id:
        return name_to_id[normalized]
    raise ValueError(f'Label kelas tidak dikenal: {value!r}. Gunakan nama kelas yang ditentukan atau ID 0-4.')

training_gdf['class'] = training_gdf[LABEL_COLUMN].map(class_id_from_value)
training_gdf['class_name'] = training_gdf['class'].map(class_names)
features = [
    ee.Feature(ee.Geometry(geometry.__geo_interface__), {'class': int(class_id)})
    for geometry, class_id in zip(training_gdf.geometry, training_gdf['class'])
]
training_fc = ee.FeatureCollection(features)

samples = selected_image.sampleRegions(
    collection=training_fc,
    properties=['class'],
    scale=20,
    tileScale=4,
    geometries=False,
)
csv_url = samples.getDownloadURL(filetype='csv')
response = requests.get(csv_url, timeout=300)
response.raise_for_status()
from io import BytesIO
df = pd.read_csv(BytesIO(response.content))
df = df.dropna(subset=bands + ['class']).copy()
df['class'] = df['class'].astype(int)
if df.empty:
    raise ValueError('Tidak ada piksel training valid setelah masking awan. Periksa lokasi poligon/tanggal citra.')
print(f'{len(df):,} sampel piksel berhasil diekstrak.')
display(df.head())
display(df['class'].map(class_names).value_counts().rename_axis('kelas').to_frame('jumlah_piksel'))

## 3. Tahap "Klasifikasi" (Machine Learning)

Random Forest menggunakan 300 pohon dan bobot kelas seimbang. Data dipisahkan secara stratified agar kelima kelas terwakili pada data uji. Karena pemisahan acak dilakukan per piksel, metrik dapat terlalu optimistis bila piksel berdekatan berasal dari poligon yang sama; untuk validasi final, pisahkan poligon atau wilayah secara spasial. Digitasi harus mencakup variasi lokasi/musim dan evaluasi perlu memperhatikan confusion matrix serta recall tiap kelas.

**Peran Band 11 (SWIR):** SWIR peka terhadap kandungan air tanah dan tajuk. Mangrove di zona pasang-surut umumnya lebih basah daripada hutan daratan sehingga pola reflektansi B11 dapat membantu memisahkan keduanya. Ini bukan aturan mutlak: kelembapan musiman, bayangan, dan kualitas label dapat menimbulkan tumpang tindih, sehingga sampel lapangan/digitasi yang representatif sangat penting.

In [ ]:
X = df[bands]
y = df['class']
class_counts = y.value_counts().reindex(class_names, fill_value=0)
if (class_counts < 4).any():
    insufficient = class_counts[class_counts < 4].index.map(class_names).tolist()
    raise ValueError(f'Setiap kelas perlu minimal 4 piksel untuk split stratified ini. Periksa: {insufficient}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    max_features='sqrt',
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

labels = list(class_names.keys())
print('--- Laporan klasifikasi (data uji) ---')
print(classification_report(
    y_test, y_pred, labels=labels,
    target_names=[class_names[i] for i in labels], zero_division=0
))
print('Confusion matrix (baris=aktual, kolom=prediksi):')
display(pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=labels),
    index=[f'Aktual: {class_names[i]}' for i in labels],
    columns=[f'Prediksi: {class_names[i]}' for i in labels],
))
print('Kepentingan fitur:')
display(pd.Series(rf_model.feature_importances_, index=bands, name='importance').sort_values(ascending=False))

## 4. Tahap Visualisasi Peta (Folium)

Peta menggunakan Esri World Imagery sebagai base map dan satu `FeatureGroup` poligon per kelas. GeoJSON yang ditampilkan di sini adalah poligon training berlabel sebagai contoh overlay; bukan hasil klasifikasi raster. Layer kelas dimulai dalam keadaan mati, sehingga saat satu kelas dicentang hanya poligon kelas tersebut yang muncul. Untuk hasil prediksi final, ganti `training_gdf` dengan GeoDataFrame poligon prediksi yang memiliki kolom `class` berisi ID kelas 0–4.

In [ ]:
class_info = {
    0: {'name': 'Bangunan', 'color': '#e31a1c'},
    1: {'name': 'Lahan Pertanian', 'color': '#ffcc00'},
    2: {'name': 'Perairan', 'color': '#1f78b4'},
    3: {'name': 'Hutan', 'color': '#33a02c'},
    4: {'name': 'Hutan Mangrove', 'color': '#006400'},
}

# Salin file GeoJSON hasil prediksi ke prediksi.geojson untuk mengganti overlay contoh.
# File tersebut perlu memiliki kolom class (ID 0-4); jika tidak ada, contoh memakai poligon training.
prediction_path = 'prediksi.geojson'
if os.path.exists(prediction_path):
    map_gdf = gpd.read_file(prediction_path)
    if 'class' not in map_gdf.columns:
        raise ValueError("prediksi.geojson harus memiliki properti 'class' berisi ID kelas 0-4.")
    if map_gdf.crs is None:
        map_gdf = map_gdf.set_crs('EPSG:4326')
    else:
        map_gdf = map_gdf.to_crs('EPSG:4326')
    map_gdf['class'] = map_gdf['class'].map(class_id_from_value)
    map_gdf['class_name'] = map_gdf['class'].map(class_names)
    print('Menampilkan poligon dari prediksi.geojson.')
else:
    map_gdf = training_gdf.copy()
    print('Menampilkan poligon training sebagai overlay contoh, bukan prediksi raster.')

m = folium.Map(location=[-7.7, 112.5], zoom_start=8, tiles=None)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri World Imagery',
    name='Esri World Imagery',
    overlay=False,
    control=True,
).add_to(m)

for class_id, info in class_info.items():
    class_polygons = map_gdf.loc[map_gdf['class'] == class_id]
    if class_polygons.empty:
        continue
    feature_group = folium.FeatureGroup(name=info['name'], show=False)
    folium.GeoJson(
        data=json.loads(class_polygons[['class_name', 'geometry']].to_json()),
        name=info['name'],
        style_function=lambda feature, color=info['color']: {
            'color': color,
            'weight': 2,
            'fillColor': color,
            'fillOpacity': 0.45,
        },
        tooltip=folium.GeoJsonTooltip(fields=['class_name'], aliases=['Kelas:']),
    ).add_to(feature_group)
    feature_group.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
print('Peta siap. Centang satu layer kelas untuk menampilkan poligon kelas tersebut.')
m